#### Clean Data

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
%matplotlib inline 
import matplotlib.pyplot as plt
import plotly.express as px
import sklearn

In [8]:
filtered_data = pd.read_csv("../Data/Filtered_311_Dataset.csv")
filtered_data.shape

(387218, 29)

### Drop Columns that are redundant (not focus of the project)

In [11]:
drop_col_data = filtered_data.drop(columns=['create_date_utc', 'last_action_utc', 'closed_date_utc', 'cross_street', 'street', 'street_id', 'cross_street_id', 'latitude', 'longitude', 'geo_accuracy', 'request_type_id'])
drop_col_data.shape

(387218, 18)

In [13]:
cleaned_data = drop_col_data.dropna(subset=['neighborhood'])
cleaned_data.shape

(365567, 18)

In [15]:
cleaned_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 365567 entries, 91 to 387217
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   _id                365567 non-null  int64  
 1   group_id           365567 non-null  int64  
 2   num_requests       365567 non-null  int64  
 3   parent_closed      365567 non-null  object 
 4   status_name        365567 non-null  object 
 5   status_code        365567 non-null  int64  
 6   dept               363091 non-null  object 
 7   request_type_name  365567 non-null  object 
 8   create_date_et     365567 non-null  object 
 9   last_action_et     365567 non-null  object 
 10  closed_date_et     312003 non-null  object 
 11  origin             365567 non-null  object 
 12  city               365567 non-null  object 
 13  neighborhood       365567 non-null  object 
 14  census_tract       233900 non-null  float64
 15  council_district   365492 non-null  float64
 16  ward  

In [17]:
cleaned_data.isnull().sum()

_id                       0
group_id                  0
num_requests              0
parent_closed             0
status_name               0
status_code               0
dept                   2476
request_type_name         0
create_date_et            0
last_action_et            0
closed_date_et        53564
origin                    0
city                      0
neighborhood              0
census_tract         131667
council_district         75
ward                     26
police_zone              50
dtype: int64

### Adding Demographic Data

In [20]:
# Read in neighborhood data

neighborhoods = pd.read_csv('../Data/2020Census_Neighborhood_Dataset.csv')

In [22]:
# Removing columns from 2010 and that are comparisons between 2020 and 2010

to_drop = neighborhoods.columns[neighborhoods.columns.str.startswith(('2010', 'Change'))]
neighborhoods.drop(to_drop, axis=1, inplace=True)

In [24]:
# Changing Neighborhood column capitalization to make merging easier

neighborhoods = neighborhoods.rename(columns = {'Neighborhood': 'neighborhood'})

In [26]:
# Changes in dataset -> census data

# Central Business District -> Central Business District (Downtown)
# Spring Hill-City View -> 	Spring Hill-City
# Mount Oliver Borough -> Mt. Oliver
# Arlington -> Arlington - Arlington Heights (Combined)
# Arlington Heights -> 	Arlington - Arlington Heights (Combined)

recode_nbhds = {'Central Business District': 'Central Business District (Downtown)', 'Spring Hill-City View': 'Spring Hill-City',
                'Mount Oliver Borough': 'Mt. Oliver', 'Arlington': 'Arlington - Arlington Heights (Combined)',
                'Arlington Heights': 'Arlington - Arlington Heights (Combined)'}

In [28]:
# Making changes to original DataFrame to combine neighborhood data

cleaned_data['neighborhood'] = cleaned_data['neighborhood'].replace(recode_nbhds)

C:\Users\cnwoo\AppData\Local\Temp\ipykernel_13416\3624421226.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_data['neighborhood'] = cleaned_data['neighborhood'].replace(recode_nbhds)


In [30]:
# Merging both DataFrams on neighborhood columns

combineddf = pd.merge(cleaned_data, neighborhoods, how = 'left', on = 'neighborhood')

### Converting to datetime format

In [33]:
# Convert date columns to DateTime format
cleaned_data['create_date_et'] = pd.to_datetime(cleaned_data['create_date_et'], errors='coerce')
cleaned_data['last_action_et'] = pd.to_datetime(cleaned_data['last_action_et'], errors='coerce')
cleaned_data['closed_date_et'] = pd.to_datetime(cleaned_data['closed_date_et'], errors='coerce')

C:\Users\cnwoo\AppData\Local\Temp\ipykernel_13416\707828681.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_data['create_date_et'] = pd.to_datetime(cleaned_data['create_date_et'], errors='coerce')
C:\Users\cnwoo\AppData\Local\Temp\ipykernel_13416\707828681.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_data['last_action_et'] = pd.to_datetime(cleaned_data['last_action_et'], errors='coerce')
C:\Users\cnwoo\AppData\Local\Temp\ipykernel_13416\707828681.py:4: SettingWithCopyWarning: 
A v

### Filling in missing values

In [36]:
# Fill in missing value for column police_zone
# Fill values based on https://pghsafeneighborhoods.wordpress.com/wp-content/uploads/2008/06/zones-by-neighborhood.pdf

# Neighborhoods without police zone
unique_neighborhoods = cleaned_data.loc[cleaned_data['police_zone'].isna(), 'neighborhood'].unique()
print(unique_neighborhoods)

# Fill in missing values for police_zone
dict_neighborhoods_policezone = {'Windgap': 6.0, 
                                 'Overbrook': 3.0,
                                 'Mount Oliver Borough': 3.9,
                                 'Mt. Oliver': 3.0,
                                 'Swisshelm Park': 4.0,
                                 'Westwood': 6.0,
                                 'Banksville': 3.0,
                                 'Fairywood': 6.0,
                                 'Knoxville': 3.0,
                                 'Ridgemont': 3.0,
                                 'East Carnegie': 6.0,
                                 'Oakwood': 6.0
}

for neighborhood, police_zone in dict_neighborhoods_policezone.items():
    cleaned_data.loc[cleaned_data['neighborhood'] == neighborhood, 'police_zone'] = police_zone

# Check if there are still missing values
missing_values_count = cleaned_data['police_zone'].isna().sum()
print(f"Missing values in police_zone: {missing_values_count}")

['Windgap' 'Overbrook' 'Mt. Oliver' 'Swisshelm Park' 'Westwood'
 'Banksville' 'Fairywood' 'Knoxville' 'Ridgemont' 'East Carnegie'
 'Oakwood']
Missing values in police_zone: 0


In [38]:
# Fill in missing departments using codebook (from 311Codebook.csv)

codebook = pd.read_csv("../Data/311 Codebook Request Types - Codebook.csv")

# Create a mapping dictionary for Issue and Department
issue_dept_mapping = codebook.set_index('Issue')['Department'].to_dict()

# Fill missing departments in cleaned_data using the mapping
cleaned_data['dept'] = cleaned_data.apply(
    lambda row: issue_dept_mapping[row['request_type_name']] if pd.isna(row['dept']) and row['request_type_name'] in issue_dept_mapping else row['dept'],
    axis=1
)

# Remaining missing rows fall under the category 'In database, but not on 311 Web Submission Form'

C:\Users\cnwoo\AppData\Local\Temp\ipykernel_13416\3811184886.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_data['dept'] = cleaned_data.apply(


In [39]:
cleaned_data.isnull().sum()

_id                       0
group_id                  0
num_requests              0
parent_closed             0
status_name               0
status_code               0
dept                   2362
request_type_name         0
create_date_et            0
last_action_et            0
closed_date_et        53564
origin                    0
city                      0
neighborhood              0
census_tract         131667
council_district         75
ward                     26
police_zone               0
dtype: int64

In [40]:
# Check neighborhoods with missing departments as a proportion of the total requests in the neighborhood
neighborhood_counts = cleaned_data['neighborhood'].value_counts()
neighborhood_counts = neighborhood_counts[neighborhood_counts > 0]

# Calculate the proportion of missing departments for each neighborhood
proportion_missing_dept = cleaned_data[cleaned_data['dept'].isna()]['neighborhood'].value_counts() / neighborhood_counts
proportion_missing_dept = proportion_missing_dept[proportion_missing_dept > 0]

# Sort the neighborhoods by proportion of missing departments
proportion_missing_dept = proportion_missing_dept.sort_values(ascending=False)

proportion_missing_dept

neighborhood
Crawford-Roberts     0.038504
North Oakland        0.034328
East Carnegie        0.031776
Crafton Heights      0.026063
Chateau              0.024291
                       ...   
Oakwood              0.001305
Hays                 0.001241
Overbrook            0.001158
South Side Slopes    0.001095
Bon Air              0.001010
Name: count, Length: 85, dtype: float64

In [44]:
# Since the missing departments are not significant, we can drop the rows with missing departments
cleaned_data = cleaned_data.dropna(subset=['dept'])

In [46]:
cleaned_data.isnull().sum()

_id                       0
group_id                  0
num_requests              0
parent_closed             0
status_name               0
status_code               0
dept                      0
request_type_name         0
create_date_et            0
last_action_et            0
closed_date_et        52289
origin                    0
city                      0
neighborhood              0
census_tract         130159
council_district         75
ward                     25
police_zone               0
dtype: int64

In [70]:
cleaned_data['ward'] = cleaned_data['ward'].fillna(value = 0)

In [72]:
cleaned_data.isnull().sum()

_id                     0
group_id                0
num_requests            0
parent_closed           0
status_name             0
status_code             0
dept                    0
create_date_et          0
last_action_et          0
closed_date_et      29628
origin                  0
city                    0
neighborhood            0
census_tract            0
council_district       63
ward                    0
police_zone             0
category                0
dtype: int64

#### Create Columns

In [49]:
# Create a new column called Category for request types using the codebook

# Create a mapping dictionary for Issue and Category
issue_category_mapping = codebook.set_index('Issue')['Category'].to_dict()

# Create new columns called category and input values based on the mapping
cleaned_data['category'] = cleaned_data.apply(
    lambda row: issue_category_mapping[row['request_type_name']] if row['request_type_name'] in issue_category_mapping else row['request_type_name'],
    axis=1
)

# Check if there are still missing values
missing_values_count = cleaned_data['category'].isna().sum()
missing_values_count

0

In [50]:
cleaned_data.isnull().sum()

_id                       0
group_id                  0
num_requests              0
parent_closed             0
status_name               0
status_code               0
dept                      0
request_type_name         0
create_date_et            0
last_action_et            0
closed_date_et        52289
origin                    0
city                      0
neighborhood              0
census_tract         130159
council_district         75
ward                     25
police_zone               0
category                  0
dtype: int64

### Dropping rows/columns

In [54]:
# Dropping rows with missing census_tract values since it is hard to impute and there are too many missing
cleaned_data = cleaned_data.dropna(subset=['census_tract'])

In [56]:
# Dropping request_type_name and using Category instead
cleaned_data = cleaned_data.drop(columns=['request_type_name'])
cleaned_data.isnull().sum()

_id                     0
group_id                0
num_requests            0
parent_closed           0
status_name             0
status_code             0
dept                    0
create_date_et          0
last_action_et          0
closed_date_et      29628
origin                  0
city                    0
neighborhood            0
census_tract            0
council_district       63
ward                   25
police_zone             0
category                0
dtype: int64

### One hot encoding

In [59]:
cleaned_data_encoded = pd.get_dummies(cleaned_data, drop_first=True, sparse=True)

In [60]:
cleaned_data_encoded.shape

(233046, 286)

### Checking for Sparse Columns

In [64]:
# from numpy import arange
# import altair as alt
# from sklearn.feature_selection import VarianceThreshold

# data = cleaned_data_encoded.values
# X = data[:, :-1]
# y = data[:, -1]

# print(X.shape, y.shape)

# thresholds = arange(0.0, 0.55, 0.05)

# results = []
# for t in thresholds:
    
#     vt = VarianceThreshold(threshold=t)
    
#     X_sel = vt.fit_transform(X)
#     rows, cols = X_sel.shape
#     n_features = cols
#     print('Threshold=%.2f, Features=%d' % (t, n_features))
    
#     results.append(n_features)
    
# d2 = pd.DataFrame({'threshold': thresholds, 'n_features': results})
# alt.Chart(d2).mark_line().encode(
#     x='threshold',
#     y='n_features')

In [66]:
cleaned_data_encoded.to_csv("../Data/CleanedData_311_Dataset.csv", index=False)